# Risk Case: Colab Training Notebook

This notebook is designed for Google Colab and works from VSCode with Colab extension.

What it does:
- loads train/test CSV
- aggregates to policy level
- trains multiple frequency models
- trains severity model
- optimizes simple pricing parameters
- generates submission


In [1]:
!pip -q install pandas numpy scikit-learn catboost xgboost lightgbm optuna pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 27.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 33.0 MB/s eta 0:00:00


In [2]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, brier_score_loss, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split

from catboost import CatBoostClassifier, CatBoostRegressor
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

SEED = 42
np.random.seed(SEED)

print('Imports done')



Imports done


In [3]:
# Colab Drive mount (robust)
from pathlib import Path
import os

try:
    from google.colab import drive
except Exception:
    drive = None

MOUNT_DIR = '/content/gdrive'
if drive is not None:
    Path(MOUNT_DIR).mkdir(parents=True, exist_ok=True)
    if os.path.ismount(MOUNT_DIR):
        print(f'Drive already mounted at {MOUNT_DIR}')
    else:
        drive.mount(MOUNT_DIR, force_remount=True)

TRAIN_CSV = Path('/content/gdrive/MyDrive/risk_case/data/train.csv')
OUT_DIR = Path('/content/gdrive/MyDrive/risk_case/artifacts/colab_runs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional: speed mode for quick iteration.
SAMPLE_N = None  # e.g. 200000

# Internal split ratios from train.csv only
VALID_SIZE = 0.20
TEST_SIZE = 0.20  # from remaining part after valid split

# GPU / tuning controls
USE_GPU = True
GPU_DEVICES = '0'
RUN_OPTUNA = True
OPTUNA_TRIALS = 200

TARGET_CLAIM = 'is_claim'
TARGET_AMOUNT = 'claim_amount'
CONTRACT_COL = 'contract_number'
TIME_COL = 'operation_date'
PREMIUM_COL = 'premium'
PREMIUM_NET_COL = 'premium_wo_term'

TARGET_LR = 0.7
TARGET_BAND = (0.69, 0.71)

if not TRAIN_CSV.exists():
    raise FileNotFoundError(f'train.csv not found: {TRAIN_CSV}')

print('Using TRAIN_CSV:', TRAIN_CSV)
print('Using OUT_DIR  :', OUT_DIR)
print('USE_GPU       :', USE_GPU)
print('RUN_OPTUNA    :', RUN_OPTUNA, 'trials=', OPTUNA_TRIALS)



Mounted at /content/gdrive
Using TRAIN_CSV: /content/gdrive/MyDrive/risk_case/data/train.csv
Using OUT_DIR  : /content/gdrive/MyDrive/risk_case/artifacts/colab_runs
USE_GPU       : True
RUN_OPTUNA    : True trials= 200


In [4]:
train_raw = pd.read_csv(TRAIN_CSV)

if SAMPLE_N is not None and SAMPLE_N > 0 and SAMPLE_N < len(train_raw):
    train_raw = train_raw.sample(SAMPLE_N, random_state=SEED).reset_index(drop=True)

print('train_raw:', train_raw.shape)
print('claim rate:', float(pd.to_numeric(train_raw[TARGET_CLAIM], errors='coerce').fillna(0).mean()))



/tmp/ipython-input-799642808.py:1: DtypeWarning: Columns (16,28) have mixed types. Specify dtype option on import or set low_memory=False.
  train_raw = pd.read_csv(TRAIN_CSV)


train_raw: (569508, 159)
claim rate: 0.01947997218651889


In [5]:
def aggregate_policy(df: pd.DataFrame, is_train: bool) -> pd.DataFrame:
    d = df.copy()
    d[TIME_COL] = pd.to_datetime(d[TIME_COL], errors='coerce')

    num_cols = d.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in d.columns if c not in num_cols + [TIME_COL]]

    protected = {CONTRACT_COL, TARGET_CLAIM, TARGET_AMOUNT, 'claim_cnt'}
    num_feature_cols = [c for c in num_cols if c not in protected]
    cat_feature_cols = [c for c in cat_cols if c not in protected]

    agg = {}
    for c in num_feature_cols:
        if c in [PREMIUM_COL, PREMIUM_NET_COL]:
            agg[c] = 'max'
        else:
            agg[c] = 'mean'
    for c in cat_feature_cols:
        agg[c] = 'first'
    agg[TIME_COL] = 'max'

    g = d.groupby(CONTRACT_COL, dropna=False).agg(agg).reset_index()

    if is_train:
        y_agg = {TARGET_CLAIM: 'max', TARGET_AMOUNT: 'sum'}
        if 'claim_cnt' in d.columns:
            y_agg['claim_cnt'] = 'sum'
        y = d.groupby(CONTRACT_COL, dropna=False).agg(y_agg).reset_index()
        g = g.merge(y, on=CONTRACT_COL, how='left')
        g[TARGET_CLAIM] = pd.to_numeric(g[TARGET_CLAIM], errors='coerce').fillna(0).astype(int)
        g[TARGET_AMOUNT] = pd.to_numeric(g[TARGET_AMOUNT], errors='coerce').fillna(0.0).clip(lower=0.0)

    g['month'] = g[TIME_COL].dt.month.fillna(0).astype(int)
    g['quarter'] = g[TIME_COL].dt.quarter.fillna(0).astype(int)
    g['dayofweek'] = g[TIME_COL].dt.dayofweek.fillna(0).astype(int)

    return g

train_policy = aggregate_policy(train_raw, is_train=True)
print('train_policy:', train_policy.shape)



train_policy: (180635, 162)


In [6]:
for c in [PREMIUM_COL, PREMIUM_NET_COL]:
    if c in train_policy.columns:
        train_policy[c] = pd.to_numeric(train_policy[c], errors='coerce').fillna(0.0)

train_policy[TIME_COL] = pd.to_datetime(train_policy[TIME_COL], errors='coerce')

y_full = train_policy[TARGET_CLAIM].astype(int)
train_df, valid_df = train_test_split(
    train_policy,
    test_size=VALID_SIZE,
    random_state=SEED,
    stratify=y_full if y_full.nunique() > 1 else None,
)

# Extra internal test split from the remaining train part
if TEST_SIZE > 0:
    y_tmp = train_df[TARGET_CLAIM].astype(int)
    train_df, test_df = train_test_split(
        train_df,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=y_tmp if y_tmp.nunique() > 1 else None,
    )
else:
    test_df = valid_df.copy()

print('split rows -> train:', len(train_df), 'valid:', len(valid_df), 'test_internal:', len(test_df))

drop_cols = {
    CONTRACT_COL, TIME_COL, TARGET_CLAIM, TARGET_AMOUNT,
    'unique_id', 'driver_iin', 'insurer_iin', 'car_number', 'claim_cnt'
}
feature_cols = [c for c in train_df.columns if c not in drop_cols]

X_train = train_df[feature_cols].copy()
X_valid = valid_df[feature_cols].copy()
X_test_internal = test_df[feature_cols].copy()

cat_cols = [c for c in feature_cols if X_train[c].dtype == 'object' or str(X_train[c].dtype).startswith('string')]
num_cols = [c for c in feature_cols if c not in cat_cols]

for c in num_cols:
    X_train[c] = pd.to_numeric(X_train[c], errors='coerce')
    X_valid[c] = pd.to_numeric(X_valid[c], errors='coerce')
    X_test_internal[c] = pd.to_numeric(X_test_internal[c], errors='coerce')
    med = X_train[c].median()
    X_train[c] = X_train[c].fillna(med)
    X_valid[c] = X_valid[c].fillna(med)
    X_test_internal[c] = X_test_internal[c].fillna(med)

for c in cat_cols:
    X_train[c] = X_train[c].astype('string').fillna('missing').astype(str)
    X_valid[c] = X_valid[c].astype('string').fillna('missing').astype(str)
    X_test_internal[c] = X_test_internal[c].astype('string').fillna('missing').astype(str)

y_train = train_df[TARGET_CLAIM].astype(int).values
y_valid = valid_df[TARGET_CLAIM].astype(int).values
y_test_internal = test_df[TARGET_CLAIM].astype(int).values

print('Features:', len(feature_cols), 'Categorical:', len(cat_cols))



split rows -> train: 115606 valid: 36127 test_internal: 28902
Features: 153 Categorical: 10


In [7]:
# Optional heavy tuning: CatBoost classifier via Optuna (on VALID split)
import optuna


def default_cb_params():
    p = {
        'iterations': 350,
        'learning_rate': 0.05,
        'depth': 6,
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'random_seed': SEED,
        'verbose': False,
    }
    if USE_GPU:
        p['task_type'] = 'GPU'
        p['devices'] = GPU_DEVICES
    return p


def suggest_cb_params(trial: optuna.Trial):
    p = {
        'iterations': trial.suggest_int('iterations', 250, 900),
        'learning_rate': trial.suggest_float('learning_rate', 0.015, 0.15, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 30.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 0.0, 2.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 5.0),
        'border_count': trial.suggest_int('border_count', 64, 255),
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'random_seed': SEED,
        'verbose': False,
    }
    if USE_GPU:
        p['task_type'] = 'GPU'
        p['devices'] = GPU_DEVICES
    return p


BEST_CB_PARAMS = default_cb_params()
if RUN_OPTUNA and OPTUNA_TRIALS > 0:
    def objective(trial: optuna.Trial) -> float:
        params = suggest_cb_params(trial)
        model = CatBoostClassifier(**params)
        model.fit(X_train, y_train, cat_features=cat_cols)
        p_valid = model.predict_proba(X_valid)[:, 1]
        auc = roc_auc_score(y_valid, p_valid)
        return float(2 * auc - 1)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=int(OPTUNA_TRIALS), show_progress_bar=False)

    BEST_CB_PARAMS = suggest_cb_params(optuna.trial.FixedTrial(study.best_trial.params))
    # Remove search-only fields from FixedTrial reconstruction if needed
    for k in ['l2_leaf_reg', 'random_strength', 'bagging_temperature', 'border_count']:
        BEST_CB_PARAMS[k] = study.best_trial.params.get(k, BEST_CB_PARAMS.get(k))

    print('Optuna best value (gini valid):', study.best_value)
    print('Optuna best params:', study.best_trial.params)
else:
    print('Optuna skipped; using default CatBoost params.')

print('BEST_CB_PARAMS =', BEST_CB_PARAMS)



[I 2026-02-24 09:58:28,610] A new study created in memory with name: no-name-3ee472c0-d58a-435a-a10c-55d40f54ca88
Default metric period is 5 because AUC is/are not implemented for GPU
[I 2026-02-24 09:58:54,091] Trial 0 finished with value: 0.4207918930724759 and parameters: {'iterations': 819, 'learning_rate': 0.038824866326251053, 'depth': 7, 'l2_leaf_reg': 3.5468447703095873, 'random_strength': 1.9514106423722601, 'bagging_temperature': 2.728610977264906, 'border_count': 253}. Best is trial 0 with value: 0.4207918930724759.
Default metric period is 5 because AUC is/are not implemented for GPU
[I 2026-02-24 09:59:05,090] Trial 1 finished with value: 0.43874685125108703 and parameters: {'iterations': 572, 'learning_rate': 0.048236697050867264, 'depth': 5, 'l2_leaf_reg': 2.594316029822117, 'random_strength': 0.7750834503396504, 'bagging_temperature': 1.2365163569310385, 'border_count': 192}. Best is trial 1 with value: 0.43874685125108703.
Default metric period is 5 because AUC is/are 

Optuna best value (gini valid): 0.4569772484377703
Optuna best params: {'iterations': 858, 'learning_rate': 0.05165789759734374, 'depth': 6, 'l2_leaf_reg': 20.573978449973225, 'random_strength': 1.1528037065085124, 'bagging_temperature': 0.6668574225163288, 'border_count': 219}
BEST_CB_PARAMS = {'iterations': 858, 'learning_rate': 0.05165789759734374, 'depth': 6, 'l2_leaf_reg': 20.573978449973225, 'random_strength': 1.1528037065085124, 'bagging_temperature': 0.6668574225163288, 'border_count': 219, 'loss_function': 'Logloss', 'eval_metric': 'AUC', 'random_seed': 42, 'verbose': False, 'task_type': 'GPU', 'devices': '0'}


In [8]:
def eval_cls(y_true: np.ndarray, p: np.ndarray) -> dict:
    auc = roc_auc_score(y_true, p)
    return {
        'auc': float(auc),
        'gini': float(2 * auc - 1),
        'brier': float(brier_score_loss(y_true, p))
    }

results = []
pred_valid = {}
pred_test_internal = {}

# CatBoost with optional Optuna-tuned params
cb = CatBoostClassifier(**BEST_CB_PARAMS)
cb.fit(X_train, y_train, cat_features=cat_cols)
p_valid_cb = cb.predict_proba(X_valid)[:, 1]
p_test_cb = cb.predict_proba(X_test_internal)[:, 1]
results.append({'model': 'catboost', **eval_cls(y_valid, p_valid_cb)})
pred_valid['catboost'] = p_valid_cb
pred_test_internal['catboost'] = p_test_cb

# XGBoost / LightGBM on one-hot matrix
X_train_oh = pd.get_dummies(X_train, columns=cat_cols, dummy_na=False)
X_valid_oh = pd.get_dummies(X_valid, columns=cat_cols, dummy_na=False)
X_test_oh = pd.get_dummies(X_test_internal, columns=cat_cols, dummy_na=False)
X_valid_oh = X_valid_oh.reindex(columns=X_train_oh.columns, fill_value=0)
X_test_oh = X_test_oh.reindex(columns=X_train_oh.columns, fill_value=0)

xgb = XGBClassifier(
    n_estimators=260,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=SEED,
    n_jobs=2
)
xgb.fit(X_train_oh, y_train)
p_valid_xgb = xgb.predict_proba(X_valid_oh)[:, 1]
p_test_xgb = xgb.predict_proba(X_test_oh)[:, 1]
results.append({'model': 'xgboost', **eval_cls(y_valid, p_valid_xgb)})
pred_valid['xgboost'] = p_valid_xgb
pred_test_internal['xgboost'] = p_test_xgb

lgbm = LGBMClassifier(
    n_estimators=320,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=SEED,
    n_jobs=2
)
lgbm.fit(X_train_oh, y_train)
p_valid_lgbm = lgbm.predict_proba(X_valid_oh)[:, 1]
p_test_lgbm = lgbm.predict_proba(X_test_oh)[:, 1]
results.append({'model': 'lightgbm', **eval_cls(y_valid, p_valid_lgbm)})
pred_valid['lightgbm'] = p_valid_lgbm
pred_test_internal['lightgbm'] = p_test_lgbm

results_df = pd.DataFrame(results).sort_values(['gini', 'auc'], ascending=False).reset_index(drop=True)
results_df



Default metric period is 5 because AUC is/are not implemented for GPU


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 2288, number of negative: 113318
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.092475 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 18730
[LightGBM] [Info] Number of data points in the train set: 115606, number of used features: 922
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.019791 -> initscore=-3.902520
[LightGBM] [Info] Start training from score -3.902520


,model,auc,gini,brier
0,catboost,0.728489,0.456977,0.019167
1,xgboost,0.709186,0.418371,0.019244
2,lightgbm,0.692141,0.384283,0.019400


In [9]:
# Train severity model (on positive claims)
pos_train = train_df[train_df[TARGET_CLAIM] > 0].copy()
y_sev_train = np.log1p(pd.to_numeric(pos_train[TARGET_AMOUNT], errors='coerce').fillna(0.0).clip(lower=0.0).values)

X_sev_train = pos_train[feature_cols].copy()
for c in num_cols:
    X_sev_train[c] = pd.to_numeric(X_sev_train[c], errors='coerce').fillna(X_train[c].median())
for c in cat_cols:
    X_sev_train[c] = X_sev_train[c].astype('string').fillna('missing').astype(str)

sev_params = {
    'iterations': int(BEST_CB_PARAMS.get('iterations', 350) * 1.2),
    'learning_rate': float(BEST_CB_PARAMS.get('learning_rate', 0.05)),
    'depth': int(BEST_CB_PARAMS.get('depth', 6)),
    'loss_function': 'RMSE',
    'random_seed': SEED,
    'verbose': False,
}
if USE_GPU:
    sev_params['task_type'] = 'GPU'
    sev_params['devices'] = GPU_DEVICES

sev_model = CatBoostRegressor(**sev_params)
sev_model.fit(X_sev_train, y_sev_train, cat_features=cat_cols)

sev_valid = np.expm1(np.clip(sev_model.predict(X_valid), 0, 20))
sev_test_internal = np.expm1(np.clip(sev_model.predict(X_test_internal), 0, 20))

pos_valid = valid_df[TARGET_CLAIM] > 0
if pos_valid.sum() > 0:
    y_true_pos = np.log1p(pd.to_numeric(valid_df.loc[pos_valid, TARGET_AMOUNT], errors='coerce').fillna(0.0).values)
    y_pred_pos = np.log1p(np.clip(sev_valid[pos_valid.values], 0, None))
    sev_rmse_valid = float(np.sqrt(mean_squared_error(y_true_pos, y_pred_pos)))
    sev_mae_valid = float(mean_absolute_error(y_true_pos, y_pred_pos))
else:
    sev_rmse_valid, sev_mae_valid = np.nan, np.nan

pos_test = test_df[TARGET_CLAIM] > 0
if pos_test.sum() > 0:
    y_true_pos_t = np.log1p(pd.to_numeric(test_df.loc[pos_test, TARGET_AMOUNT], errors='coerce').fillna(0.0).values)
    y_pred_pos_t = np.log1p(np.clip(sev_test_internal[pos_test.values], 0, None))
    sev_rmse_test = float(np.sqrt(mean_squared_error(y_true_pos_t, y_pred_pos_t)));
    sev_mae_test = float(mean_absolute_error(y_true_pos_t, y_pred_pos_t))
else:
    sev_rmse_test, sev_mae_test = np.nan, np.nan

print({'severity_valid_rmse': sev_rmse_valid, 'severity_valid_mae': sev_mae_valid, 'severity_test_rmse': sev_rmse_test, 'severity_test_mae': sev_mae_test})



{'severity_valid_rmse': 1.04504873854517, 'severity_valid_mae': 0.8436938548845649, 'severity_test_rmse': 1.113385844186599, 'severity_test_mae': 0.8813670408890597}


In [10]:
def evaluate_pricing(df_valid: pd.DataFrame, new_premium: np.ndarray, target_lr: float = 0.7, target_band=(0.69, 0.71)):
    base = pd.to_numeric(df_valid[PREMIUM_COL], errors='coerce').fillna(0.0).clip(lower=0.0)
    prem_wo_term = pd.to_numeric(df_valid[PREMIUM_NET_COL], errors='coerce').fillna(base)
    payout = pd.to_numeric(df_valid[TARGET_AMOUNT], errors='coerce').fillna(0.0).clip(lower=0.0)

    term_ratio = np.where(base > 0, prem_wo_term / base, 1.0)
    term_ratio = np.nan_to_num(term_ratio, nan=1.0, posinf=1.0, neginf=1.0).clip(0.0, 1.0)
    new_premium_net = pd.Series(new_premium * term_ratio, index=df_valid.index)

    def safe_ratio(a, b):
        return float(a / b) if b > 0 else 0.0

    g1 = new_premium <= base.values
    g2 = ~g1

    lr_total = safe_ratio(float(payout.sum()), float(new_premium_net.sum()))
    lr_g1 = safe_ratio(float(payout[g1].sum()), float(new_premium_net[g1].sum()))
    lr_g2 = safe_ratio(float(payout[g2].sum()), float(new_premium_net[g2].sum()))
    share_g1 = float(np.mean(g1))

    violations = int((new_premium < 0).sum() + (new_premium > 3.0 * base.values).sum())
    distance = abs(lr_total - target_lr)
    in_target = bool(target_band[0] <= lr_total <= target_band[1])
    score = -distance - 0.7 * abs(lr_g1 - target_lr) - 0.7 * abs(lr_g2 - target_lr) + 0.15 * share_g1 - 2.0 * violations
    if in_target:
        score += 0.5

    return {
        'lr_total': lr_total,
        'lr_group1': lr_g1,
        'lr_group2': lr_g2,
        'share_group1': share_g1,
        'violations': violations,
        'distance_to_target': distance,
        'in_target': in_target,
        'policy_score': float(score),
    }

best_model = results_df.iloc[0]['model']
p_valid = pred_valid[best_model]
p_test_internal = pred_test_internal[best_model]

exp_loss_valid = p_valid * sev_valid

# Relative-to-base pricing formulation to control scale
base_valid = pd.to_numeric(valid_df[PREMIUM_COL], errors='coerce').fillna(0.0).clip(lower=0.0).values
rel_loss = exp_loss_valid / np.clip(base_valid, 1.0, None)

alphas = np.linspace(-0.25, 0.25, 81)
betas = np.linspace(0.0, 1.2, 61)

best = None
for beta in betas:
    for alpha in alphas:
        multiplier = np.clip(1.0 + alpha + beta * rel_loss, 0.4, 3.0)
        new_premium = base_valid * multiplier
        m = evaluate_pricing(valid_df, new_premium, target_lr=TARGET_LR, target_band=TARGET_BAND)

        rank_key = (
            1 if m['violations'] == 0 else 0,
            1 if m['in_target'] else 0,
            -abs(m['lr_total'] - TARGET_LR),
            m['policy_score'],
        )
        if best is None or rank_key > best['_rank_key']:
            best = {'alpha': float(alpha), 'beta': float(beta), **m, '_rank_key': rank_key}

best.pop('_rank_key', None)

internal_test_cls = eval_cls(y_test_internal, p_test_internal)
print('Best model (selected on valid):', best_model)
print('Internal test classification:', internal_test_cls)
print(json.dumps(best, ensure_ascii=False, indent=2))



Best model (selected on valid): catboost
Internal test classification: {'auc': 0.709571076646615, 'gini': 0.41914215329323, 'brier': 0.01919339306317857}
{
  "alpha": 0.9468354430379747,
  "beta": 0.8,
  "lr_total": 2.292740845725691,
  "lr_group1": 2.349453554565308,
  "lr_group2": 2.2853899522309056,
  "share_group1": 0.3545270850056744,
  "violations": 2865,
  "distance_to_target": 1.592740845725691,
  "in_target": false,
  "policy_score": -5733.803952237732
}


In [11]:
# Save internal evaluation artifacts (train.csv only workflow)
internal_test_predictions = pd.DataFrame({
    CONTRACT_COL: test_df[CONTRACT_COL].values,
    'p_claim': p_test_internal,
    'expected_severity': sev_test_internal,
    'expected_loss': p_test_internal * sev_test_internal,
})

pred_path = OUT_DIR / 'internal_test_predictions.csv'
metrics_path = OUT_DIR / 'metrics_colab.json'
summary_path = OUT_DIR / 'model_results_colab.csv'

internal_test_predictions.to_csv(pred_path, index=False)
results_df.to_csv(summary_path, index=False)
# pricing on internal test using chosen alpha/beta
base_test_internal = pd.to_numeric(test_df[PREMIUM_COL], errors='coerce').fillna(0.0).clip(lower=0.0).values
exp_loss_test_internal = p_test_internal * sev_test_internal
rel_loss_test = exp_loss_test_internal / np.clip(base_test_internal, 1.0, None)
multiplier_test = np.clip(1.0 + best['alpha'] + best['beta'] * rel_loss_test, 0.4, 3.0)
new_premium_test_internal = base_test_internal * multiplier_test
pricing_internal_test = evaluate_pricing(test_df, new_premium_test_internal, target_lr=TARGET_LR, target_band=TARGET_BAND)

with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump({
        'best_model_selected_on_valid': best_model,
        'classification_valid': results_df.to_dict(orient='records'),
        'classification_internal_test': internal_test_cls,
        'pricing_valid': best,
        'pricing_internal_test': pricing_internal_test,
        'severity': {
            'valid_rmse': sev_rmse_valid,
            'valid_mae': sev_mae_valid,
            'test_rmse': sev_rmse_test,
            'test_mae': sev_mae_test,
        },
        'note': 'No test_final submission generated in this notebook version.'
    }, f, ensure_ascii=False, indent=2)

print('Saved:', pred_path)
print('Saved:', metrics_path)
print('Saved:', summary_path)
internal_test_predictions.head()




Saved: /content/gdrive/MyDrive/risk_case/artifacts/colab_runs/internal_test_predictions.csv
Saved: /content/gdrive/MyDrive/risk_case/artifacts/colab_runs/metrics_colab.json
Saved: /content/gdrive/MyDrive/risk_case/artifacts/colab_runs/model_results_colab.csv


,contract_number,p_claim,expected_severity,expected_loss
0,ffd144faa0b22949a9556cbd022f9b2a423aef80c0adb8...,0.016703,2.115881e+06,35341.316365
1,9e9869a72f50cf1cb7746d261b5d3a49c008ae5c80cb8f...,0.003123,1.270985e+06,3969.149109
2,1350f3419e0caa780911b841d9c6c23e359ec095aad244...,0.003420,9.589015e+05,3279.355489
3,aabaac901caf3fc895dc1f114e958af7cde48a891c51bb...,0.003733,1.095123e+06,4088.232599
4,93f687d8e491a68bb0de491f893a6e0e963be9edeb593a...,0.011171,1.087276e+06,12145.798419
